# Notebook 00 — End-to-End Pipeline Test

This notebook scans every Excel file in the samples folder, classifies each sheet, and shows whether it is admissible for the normalized market dataset or must be reported and ignored.

In [ ]:
from pathlib import Path
import pandas as pd

from src.parsers.parser_factory import SheetKind, detect_sheet_family


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'samples').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not find the repository root from the current notebook location.')


repo_root = find_repo_root(Path.cwd().resolve())
samples_dir = repo_root / 'samples'
excel_files = sorted(samples_dir.glob('*.xlsx'))

print('Repository root:', repo_root)
print('Excel files found:')
for path in excel_files:
    print('-', path.name)

Repository root: /home/yass/Desktop/DSS_CMR
Excel files found:
- Compo_All_Indices_20260731_copy.xlsx
- Data_sheet_without_legend.xlsx
- Données Marché Boursier_Projet_IA_copy.xlsx


In [ ]:
def decide_sheet_action(family: SheetKind) -> str:
    if family == SheetKind.MARKET_FAMILY_A:
        return 'parse_and_include'
    if family == SheetKind.MARKET_FAMILY_B:
        return 'report_and_ignore'
    if family == SheetKind.INDEX_COMPOSITION:
        return 'skip_market_pipeline'
    return 'manual_review'


for workbook_path in excel_files:
    print(f'\n=== {workbook_path.name} ===')
    try:
        xls = pd.ExcelFile(workbook_path)
        for sheet_name in xls.sheet_names:
            raw = pd.read_excel(workbook_path, sheet_name=sheet_name, header=None)
            family = detect_sheet_family(raw)
            action = decide_sheet_action(family)
            print(f'[{sheet_name}] -> {family.value} | {action}')
            if action == 'parse_and_include':
                print('  -> admissible for normalization into the unified market dataset')
            elif action == 'report_and_ignore':
                print('  -> valid Family B sheet, report it, then exclude it from normalization')
            elif action == 'skip_market_pipeline':
                print('  -> index-composition sheet, keep independent for dynamic filtering')
            else:
                print('  -> unable to classify confidently; review manually')
    except Exception as exc:
        print(f'Error reading workbook: {exc}')


=== Compo_All_Indices_20260731_copy.xlsx ===
[MASI] -> Family A | parse_and_include
  -> admissible for normalization into the unified market dataset
[Sector Indices] -> Family A | parse_and_include
  -> admissible for normalization into the unified market dataset
[MASI 20] -> Family A | parse_and_include
  -> admissible for normalization into the unified market dataset
[MASI ESG] -> Family A | parse_and_include
  -> admissible for normalization into the unified market dataset
[MASI Mid and Small Cap] -> Family A | parse_and_include
  -> admissible for normalization into the unified market dataset

=== Data_sheet_without_legend.xlsx ===
[Feuille 1] -> Family B | report_and_ignore
  -> valid Family B sheet, report it, then exclude it from normalization

=== Données Marché Boursier_Projet_IA_copy.xlsx ===
[Data] -> Family A | parse_and_include
  -> admissible for normalization into the unified market dataset
[Cours] -> Family A | parse_and_include
  -> admissible for normalization into 

## Inclusion contract

The normalization engine should not rely on a separate merged master table for every parsed worksheet. Instead, each parser should emit an explicit inclusion decision such as `include_in_market_dataset = True` for admissible Family A market sheets and `False` for Family B sheets like Data.

This keeps the design generic, avoids a hardcoded allowlist in the merge stage, and ensures that only parser-admitted market tables participate in the unified market dataset. The index-composition dataset stays independent and is loaded later only by the dynamic filtering stage.